# Vision-R1 CoT Audit & Consolidation

Audit the Vision-R1-cold mulberry split, sanity-check images and CoT structure, then consolidate to a single JSONL with parsed `<think>/<answer>` for downstream training.

**Sources**
- Annotations: `raw/vision_r1_cold/vision_r1_mulberry_sft_full.json` (Vision-R1's re-annotated CoT, ~198k rows)
- Images: `raw/mulberry/mulberry_images/` (extracted from Mulberry-SFT, symlinked into `raw/vision_r1_cold/mulberry_images/`)

**Output**: `processed/vision_r1/all.jsonl`

LLaVA-CoT split is intentionally dropped — we did not download those images.

In [ ]:
import json
import re
import random
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

DATA_ROOT = Path('/mnt/data4/shasta/amar.amarjyoti/research_data')
VR1_DIR   = DATA_ROOT / 'raw' / 'vision_r1_cold'
MULB_DIR  = DATA_ROOT / 'raw' / 'mulberry'
OUT_DIR   = DATA_ROOT / 'processed' / 'vision_r1'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MULB_JSON = VR1_DIR / 'vision_r1_mulberry_sft_full.json'
IMAGE_ROOT = VR1_DIR  # JSON image paths are relative to this (mulberry_images symlink lives here)

print('Loading mulberry annotations ...')
mulb = json.loads(MULB_JSON.read_text())
print(f'  {len(mulb):,} rows')

## 1. Schema audit
Show keys, types, and a representative sample.

In [ ]:
key_counts = Counter()
for r in mulb:
    key_counts.update(r.keys())
print('Top-level key coverage:')
for k, c in key_counts.most_common():
    print(f'  {k:20s} {c:>8,} ({100*c/len(mulb):.1f}%)')

print('\n--- sample record ---')
sample = mulb[0]
print(json.dumps({k: (v if k != 'conversations' else f'[{len(v)} turns]') for k, v in sample.items()}, indent=2))
print('\nconversations[0]:'); print(json.dumps(sample['conversations'][0], indent=2)[:800])
print('\nconversations[1] (truncated):'); print(json.dumps(sample['conversations'][1], indent=2)[:1200])

## 2. Conversation structure check
Every row should have exactly one human + one gpt turn, with `<think>...</think><answer>...</answer>` in the gpt turn.

In [ ]:
THINK_RE  = re.compile(r'<think>(.*?)</think>',   re.DOTALL)
ANSWER_RE = re.compile(r'<answer>(.*?)</answer>', re.DOTALL)

issues = Counter()
parsed_rows = []  # populated here so we can reuse below
for r in mulb:
    convs = r.get('conversations', [])
    if len(convs) != 2:                              issues['wrong_turn_count']  += 1
    if not convs or convs[0].get('from') != 'human': issues['bad_first_speaker'] += 1
    if len(convs) < 2 or convs[1].get('from') != 'gpt': issues['bad_second_speaker'] += 1
    if len(convs) >= 2:
        gpt_text = convs[1].get('value', '')
        t = THINK_RE.search(gpt_text)
        a = ANSWER_RE.search(gpt_text)
        if not t: issues['missing_think']  += 1
        if not a: issues['missing_answer'] += 1
        parsed_rows.append({
            'id': r.get('id'),
            'image': r.get('image'),
            'question': convs[0].get('value', ''),
            'raw': gpt_text,
            'reasoning': t.group(1).strip() if t else None,
            'answer':    a.group(1).strip() if a else None,
        })

print(f'Total rows: {len(mulb):,}')
print('Structural issues (counts):')
for k, v in issues.most_common():
    print(f'  {k:24s} {v:>8,}')
print(f'Cleanly parsed (<think>+<answer>): {sum(1 for p in parsed_rows if p["reasoning"] and p["answer"]):,}')

## 3. Source distribution
Bucket by image-path prefix (ai2d, chartqa, geoqa, clevr-math, ...).

In [ ]:
def src_of(p: str) -> str:
    # paths look like 'mulberry_images/<source>/...'
    parts = (p or '').split('/')
    return parts[1] if len(parts) > 1 else '<unknown>'

src_counts = Counter(src_of(r['image']) for r in mulb)
df_src = pd.DataFrame(src_counts.most_common(), columns=['source', 'count'])
df_src['pct'] = 100 * df_src['count'] / df_src['count'].sum()
display(df_src)

## 4. Length distributions
Whitespace-token counts for question / reasoning / answer.

In [ ]:
df = pd.DataFrame(parsed_rows)
df['q_tok']    = df['question'].fillna('').str.split().str.len()
df['cot_tok']  = df['reasoning'].fillna('').str.split().str.len()
df['ans_tok']  = df['answer'].fillna('').str.split().str.len()
display(df[['q_tok', 'cot_tok', 'ans_tok']].describe(percentiles=[.5, .9, .95, .99]).round(1))

## 5. Image existence audit
Stratified sample of 2000 rows; check if the image file exists on disk. Report missing-rate per source.

In [ ]:
random.seed(0)
SAMPLE_PER_SRC = 2000 // max(len(src_counts), 1)
by_src = defaultdict(list)
for r in mulb:
    by_src[src_of(r['image'])].append(r)

missing = Counter(); checked = Counter()
missing_examples = []
for s, rows in by_src.items():
    sample = random.sample(rows, min(SAMPLE_PER_SRC, len(rows)))
    for r in sample:
        checked[s] += 1
        if not (IMAGE_ROOT / r['image']).exists():
            missing[s] += 1
            if len(missing_examples) < 10:
                missing_examples.append(r['image'])

df_img = pd.DataFrame([
    {'source': s, 'checked': checked[s], 'missing': missing[s], 'miss_pct': 100*missing[s]/max(checked[s], 1)}
    for s in sorted(checked)
]).sort_values('miss_pct', ascending=False)
display(df_img)
print('First missing-image examples:')
for p in missing_examples: print('  ', p)

## 6. Consolidate → JSONL
Write one row per cleanly parsed sample with absolute image path and pre-parsed `reasoning`/`answer`. Drop rows that:
- fail the `<think>` or `<answer>` parse, OR
- have a missing image on disk.

In [ ]:
OUT_PATH = OUT_DIR / 'all.jsonl'
n_kept = n_dropped_parse = n_dropped_image = 0
with OUT_PATH.open('w') as f:
    for r, p in zip(mulb, parsed_rows):
        if not (p['reasoning'] and p['answer']):
            n_dropped_parse += 1; continue
        img_abs = (IMAGE_ROOT / r['image']).resolve()
        if not img_abs.exists():
            n_dropped_image += 1; continue
        out = {
            'id':         r['id'],
            'source':     'vision_r1_mulberry',
            'subsource':  src_of(r['image']),
            'image_path': str(img_abs),
            'image_rel':  r['image'],
            'question':   p['question'],
            'raw':        p['raw'],
            'parsed':     {'reasoning': p['reasoning'], 'answer': p['answer']},
        }
        f.write(json.dumps(out) + '\n')
        n_kept += 1

print(f'Wrote {n_kept:,} rows to {OUT_PATH}')
print(f'Dropped (parse failure): {n_dropped_parse:,}')
print(f'Dropped (missing image): {n_dropped_image:,}')
print(f'Output size: {OUT_PATH.stat().st_size / 1e6:.1f} MB')

## 7. Final summary

In [ ]:
summary = pd.DataFrame([
    {'metric': 'input rows',           'value': len(mulb)},
    {'metric': 'parse failures',       'value': n_dropped_parse},
    {'metric': 'missing images (full sweep above is sampled — full sweep happens during write)', 'value': n_dropped_image},
    {'metric': 'consolidated rows',    'value': n_kept},
    {'metric': 'output JSONL',         'value': str(OUT_PATH)},
])
display(summary)